In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:


from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

files

In [ ]:
documents = []

for file in files:
    doc = file.parse()
    print(doc)
    documents.append(doc)

print(documents)


In [ ]:
import requests
from minsearch import Index

index=Index(
    text_fields=['content','filename'],
    keyword_fields=['filename']
)
index.fit(documents)


In [ ]:
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv()
openai_client = OpenAI()

def llm(prompt):
    response=openai_client.responses.create(
        model="gpt-3.5-tubo",
        input=prompt
    )


def search(index,query, num_results=1):
        #filter_dict = {'filename': 'llm-zoomcamp'}

        return index.search(
            query,
            num_results=num_results
            #filter_dict=filter_dict
        )


question="How does the agentic loop keep calling the model until it stops?"
print(search(index,question))

In [ ]:
INSTRUCTIONS = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
'''

PROMPT_TEMPLATE = '''
QUESTION: {question}

CONTEXT:
{context}
'''.strip()


class RAGBase:

    def __init__(
        self,
        index,
        llm_client,
        instructions=INSTRUCTIONS,
        prompt_template=PROMPT_TEMPLATE,
        course='llm-zoomcamp',
        model='gpt-5.4-mini'
    ):
        self.index = index
        self.llm_client = llm_client
        self.instructions = instructions
        self.course = course
        self.prompt_template = prompt_template
        self.model = model

  
    def search(self, query, num_results=5):
        return self.index.search(
            query,
            num_results=num_results
            #filter_dict=filter_dict
        )


    def build_context(self, search_results):
        lines = []

        for doc in search_results:
            lines.append(doc)

        return lines

    def build_prompt(self, query, search_results):
        context = self.build_context(search_results)
        return self.prompt_template.format(
            question=query, context=context
        )

    def llm(self, prompt):
        input_messages = [
            {'role': 'developer', 'content': self.instructions},
            {'role': 'user', 'content': prompt}
        ]

        response = self.llm_client.responses.create(
            model=self.model,
            input=input_messages
        )

        return response.usage.input_tokens

    def rag(self, query):
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        answer = self.llm(prompt)
        return answer
    
class OlammaRAG(RAGBase):
  
    def llm(self, prompt):
        input_messages = [
            {'role': 'system', 'content': self.instructions},
            {'role': 'user', 'content': prompt}
        ]

        response = self.llm_client.responses.create(
            model=self.model,
            input=input_messages
        )
        return 1
    

    def rag(self, query):
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        answer = self.llm(prompt)
        return answer
    
assistant = RAGBase(index, openai_client)

print(assistant.rag("How does the agentic loop keep calling the model until it stops?"))

In [ ]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
index.fit(chunks)

assistant = RAGBase(index, openai_client)

print(assistant.rag("How does the agentic loop keep calling the model until it stops?"))

In [ ]:
!uv add toyaikit

In [ ]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [ ]:

def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5
    )

In [ ]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [ ]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)
INSTRUCTIONS="You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering."
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=INSTRUCTIONS,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [ ]:
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)